# Production ML Infrastructure Tutorial

**Experiment tracking, hyperparameter tuning, and model registry**

This tutorial walks through the production ML tools in QuantStrata:

1. **Experiment tracking** – In-memory, MLflow, and Weights & Biases
2. **Hyperparameter tuning** – Search spaces, Optuna, and pruning
3. **Model registry** – Versioning and promotion to production
4. **Orchestrator pipeline** – Running tuning from config

**References:** `docs/reference/machine_learning/production_ml.md`, `docs/guides/machine_learning/experiment_tracking.md`, `hyperparameter_tuning.md`

---

## 1. Setup and imports

In [ ]:
import sys
sys.path.insert(0, '../../..')

import numpy as np
from pathlib import Path

np.random.seed(42)
print("Imports and path set.")

## 2. Experiment tracking

Use **InMemoryTracker** for notebooks and tests (no server). Log params, metrics, and artifacts.

In [ ]:
from src.machine_learning.core.tracking import InMemoryTracker

tracker = InMemoryTracker(experiment_name="ml_production_tutorial")

with tracker.start_run("run_1"):
    tracker.log_params({"learning_rate": 0.001, "hidden_units": 128})
    for step in range(5):
        loss = 0.5 - step * 0.05 + np.random.randn() * 0.02
        tracker.log_metrics({"loss": float(loss)}, step=step)
    tracker.log_metrics({"final_loss": 0.25}, step=5)

runs = tracker.get_all_runs()
best = tracker.get_best_run("final_loss", minimize=True)
print(f"Runs: {len(runs)}")
print(f"Best run params: {best.params}")
print(f"Best final_loss: {best.metrics.get('final_loss')}")

## 3. Hyperparameter tuning with Optuna

Define a **SearchSpace**, an objective function, and run **run_optuna_tuning** with optional pruning.

In [ ]:
from src.machine_learning.tuning import SearchSpace, MedianPruner, run_optuna_tuning

def dummy_objective(config, trial):
    """Minimize a simple function of lr and hidden_units; report intermediate for pruning."""
    lr = config["learning_rate"]
    hu = config["hidden_units"]
    loss = 0.1 * (np.log10(lr) + 3) ** 2 + 0.001 * (hu - 128) ** 2 / 1000
    for epoch in range(10):
        intermediate = loss * (1 - epoch / 15) + 0.02 * np.random.randn()
        trial.report(intermediate, epoch)
        if trial.should_prune():
            import optuna
            raise optuna.TrialPruned()
    return loss + 0.01 * np.random.randn()

space = (
    SearchSpace()
    .add_float("learning_rate", 1e-4, 1e-2, log=True)
    .add_int("hidden_units", 32, 256)
)

result = run_optuna_tuning(
    objective_fn=dummy_objective,
    search_space=space,
    n_trials=15,
    direction="minimize",
    pruner=MedianPruner(n_startup_trials=3, n_warmup_steps=2),
    seed=42,
)

print(f"Best config: {result.best_config}")
print(f"Best score: {result.best_score:.6f}")
print(f"Completed trials: {result.n_completed}, Pruned: {result.n_pruned}")

## 4. Model registry

Register a model version with metadata and promote it to staging or production.

In [ ]:
from src.machine_learning.registry import ModelRegistry, ModelStage

# Use a directory that exists (create a dummy artifact dir for the tutorial)
artifact_dir = Path("./artifacts/ml_tutorial_pricer")
artifact_dir.mkdir(parents=True, exist_ok=True)
(artifact_dir / "model_info.json").write_text('{"name": "tutorial_pricer"}')

registry = ModelRegistry()  # or create_registry(base_path=...)
version = registry.register_model(
    name="option_pricer",
    model_path=artifact_dir,
    params={"framework": "keras", "input_dim": 10},
    metrics={"val_loss": 0.01},
    description="Tutorial v1",
)
print(f"Registered: {version} -> stage {version.stage}")

registry.promote_to_stage("option_pricer", version.version, ModelStage.STAGING)
versions = registry.list_versions("option_pricer")
print(f"After promote: versions = {[(v.version, v.stage) for v in versions]}")

## 5. Running the hyperparameter tuning pipeline

The orchestrator pipeline `ml.hyperparameter_tuning` is built with **create_hyperparameter_tuning_pipeline** (config + objective). For a config-driven run from YAML/CLI, use the example script **examples/pipelines/run_hyperparameter_tuning.py**.

Here we only show the pattern: build pipeline with a small search space and run it via PipelineRunner.

In [ ]:
# Optional: run the tuning pipeline (requires HyperparameterTuningConfig + objective)
# from src.orchestrator.pipelines.ml.hyperparameter_tuning import (
#     create_hyperparameter_tuning_pipeline,
#     HyperparameterTuningConfig,
# )
# config = HyperparameterTuningConfig(
#     search_space_config={...},
#     n_trials=20,
#     direction="minimize",
# )
# pipeline = create_hyperparameter_tuning_pipeline(config, objective_fn=my_objective)
# Then run with PipelineRunner and Context.

print("See examples/pipelines/run_hyperparameter_tuning.py for a full pipeline example.")

---
## Summary

- **Experiment tracking:** Use `InMemoryTracker` in notebooks; use `MLflowTracker` or `WandBTracker` for shared experiments.
- **Hyperparameter tuning:** Define `SearchSpace`, pass an objective to `run_optuna_tuning`, optionally with pruners.
- **Model registry:** Register artifacts with `ModelRegistry`, promote versions to staging/production.
- **Next:** Integrate tracking and registry into your training scripts and orchestrator pipelines.